# **ST 554 Spring 2026 Project Two**
## Created by Cody Ashby on March 17, 2026
### *Testing a Customized Class on a Data Set (Part I)*

To start things off, we'll use a class that performs a quality check for a Spark SQL style dataframe. A handful of modules will need to be imported before implementing this class.

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

Before proceeding, we'll need to initiate a Spark session.

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('ST 554 Project Two').config('spark.sql.ansi.enabled','false').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 22:56:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 22:56:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Now, we'll import the Python file that contains the class so we can do our quality check.

In [3]:
import SparkDataQualityCheck

We'll use the method that reads in a csv file, namely the air quality data set from our first project, to try this out. We'll also use `pandas` to read in the same file.

In [4]:
pandas_data=pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
data=spark.createDataFrame(pandas_data)

Let's create an instance that allows us to read the csv file from the url address below.

In [5]:
air_data=SparkDataQualityCheck.SparkDataCheck("https://www4.stat.ncsu.edu/online/datasets/air.csv")

An alternative may include downloading the csv file and uploading it to the JupyterHub directory. We can then apply the associated class method to read the data in.

In [6]:
spark_air_data=air_data.read_csv(spark,"air.csv")

Now, we'll test a small handful of the functions defined in the class. Let's try a value check on the temperature.

In [7]:
spark_air_data.value_check("T",0,30)

ERROR! This is not a numeric column!
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|

26/03/26 22:57:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-cashby@ncsu.edu/ST%20554%20Project%20Two/air.csv


Hmm, that was strange. We know for certain that, with the exception for Date and Time, that nearly all of the columns are a numeric type.
Chances are something was typed in wrong when defining this function. I normally would've tried this function with just one number instead of two, like below.

In [8]:
#Here's another example of using the value_check function, but with only one number provided.
spark_air_data.value_check("NO2(GT)",123)

ERROR! This is not a numeric column!
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|

26/03/26 22:57:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-cashby@ncsu.edu/ST%20554%20Project%20Two/air.csv


Let's try using a different function and we'll see what happens?

In [10]:
spark_air_data.missing_check("NOx(GT)")

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `column` is not supported.

Huh. This is quite perplexing for me. I'll try another function just for kicks.

In [12]:
spark_air_data.level_check("Date",["3/10/2004","4/10/2004","5/10/2004","6/10/2004"])

ERROR! This is not a string column!
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2

26/03/26 23:00:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-cashby@ncsu.edu/ST%20554%20Project%20Two/air.csv


Okay, it's clear that I typed in something wrong when defining this class. Maybe we should read in the `pandas` dataframe to see if things will go differently?

In [13]:
pandas_air_data=air_data.read_pandas(spark,pandas_data)

Alright, nothing went wrong there. Let's try a function using this approach.

In [14]:
pandas_air_data.value_check("CO(GT)",1,1.5)

ERROR! This is not a numeric column!
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2| 

How embarassing! Somethiing *definitely* went awry when constructing the `.py` file that contained the class. 

### *Practicing pandas-on-spark and Spark SQL on NFL Data (Part II)*

In this portion, we'll utilize `pandas-on-spark` and Spark SQL techniques to perform some basic data analysis on an NFL data set. We'll use the former to read in this data set first.

In [15]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('ST 554 Project Two').config('spark.sql.ansi.enabled','false').getOrCreate()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


Now, we'll use `pandas` to read in the data set.

In [16]:
NFL_data=pd.read_csv("weekly_nfl_data.csv")
spark_NFL_data=ps.from_pandas(NFL_data)
spark_NFL_data.head()

26/03/26 23:10:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Below is a list of the column names.

In [17]:
spark_NFL_data.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Now, we'll collect some statistics involving quarterback players during the regular season from 2005 to 2023. Namely, we're interested in who the quarterback players are (of course), the season, the week in the season, and the number of completions, attempts, passing yards, passing touchdowns, and interceptions.

To begin with, we'll construct a table that is grouped by player and season and find the total and average for the remaining variables above. The first 20 observations are given below.

In [19]:
QB_table=spark_NFL_data.loc[((spark_NFL_data.position == "QB") & (spark_NFL_data.season > 2004) & (spark_NFL_data.season_type == "REG")), ['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']]
QB_stats=QB_table.groupby(['player_display_name','season']).agg(['sum','mean'])[['completions','attempts','passing_yards','passing_tds','interceptions']]
QB_stats.head(20)

completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333
Jeff Garcia         2005           102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000
Doug Flutie         2005             5   1.250000       10   2.500000          29.0    7.250000           0  0.000000           0.0  0.000000
Brett Favre         2005           372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500
Todd Bouman         2005            68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000
Brad Johnson        2005           183  16.636364      293  26.636364        1885.0  171.363636          12  1.090909           4.0  0.363636
Brian Griese        2005           112  18.666667      174  29.000000        1136.0  189.333333           7  1.166667           7.0  1.166667
Gus Frerotte        2005           257  16.062500      494  30.875000        2996.0  187.250000          18  1.125000          13.0  0.812500
Kelly Holcomb       2005           155  15.500000      230  23.000000        1509.0  150.900000          10  1.000000           8.0  0.800000
Mark Brunell        2005           262  16.375000      454  28.375000        3050.0  190.625000          23  1.437500          10.0  0.625000
Jay Fiedler         2005             8   4.000000       13   6.500000         107.0   53.500000           1  0.500000           0.0  0.000000
Koy Detmer          2005            32   6.400000       56  11.200000         238.0   47.600000           0  0.000000           3.0  0.600000
Jake Delhomme       2005           262  16.375000      435  27.187500        3421.0  213.812500          24  1.500000          16.0  1.000000
Aaron Brooks        2005           240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692
Trent Dilfer        2005           199  18.090909      333  30.272727        2321.0  211.000000          11  1.000000          12.0  1.090909
Trent Green         2005           317  19.812500      507  31.687500        4014.0  250.875000          17  1.062500          10.0  0.625000

Obviously, there are way more than 20 observations in this table. Let's see the dimensions of the table to get a clear picture of how many total observations there are.

In [20]:
QB_stats.shape

(1456, 10)

We can also create some new variables pertaining to the completion percentage as well as the ratio between touchdowns and interceptions.

In [21]:
QB_stats['completion_percentage']=QB_table.groupby(['player_display_name','season']).completions.sum()/QB_table.groupby(['player_display_name','season']).attempts.sum()
QB_stats['td_int_ratio']=QB_table.groupby(['player_display_name','season']).passing_tds.sum()/QB_table.groupby(['player_display_name','season']).interceptions.sum()

Our final table below contains the grouped sums along with the newly defined variables. Here are the first 40 entries.

In [22]:
QB_stats.head(40)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000              0.888889          inf
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Tony Banks          2005            14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000              0.560000     0.500000
Charlie Batch       2005            23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333              0.638889     1.000000
Jeff Garcia         2005           102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000              0.589595     0.500000
Doug Flutie         2005             5   1.250000       10   2.500000          29.0    7.250000           0  0.000000           0.0  0.000000              0.500000          NaN
Brett Favre         2005           372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500              0.612850     0.689655
Todd Bouman         2005            68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000              0.557377     0.285714
Brad Johnson        2005           183  16.636364      293  26.636364        1885.0  171.363636          12  1.090909           4.0  0.363636              0.624573     3.000000
Brian Griese        2005           112  18.666667      174  29.000000        1136.0  189.333333           7  1.166667           7.0  1.166667              0.643678     1.000000
Gus Frerotte        2005           257  16.062500      494  30.875000        2996.0  187.250000          18  1.125000          13.0  0.812500              0.520243     1.384615
Kelly Holcomb       2005           155  15.500000      230  23.000000        1509.0  150.900000          10  1.000000           8.0  0.800000              0.673913     1.250000
Mark Brunell        2005           262  16.375000      454  28.375000        3050.0  190.625000          23  1.437500          10.0  0.625000              0.577093     2.300000
Jay Fiedler         2005             8   4.000000       13   6.500000         107.0   53.500000           1  0.500000           0.0  0.000000              0.615385          inf
Koy Detmer          2005            32   6.400000       56  11.200000         238.0   47.600000           0  0.000000           3.0  0.600000              0.571429     0.000000
Jake Delhomme       2005           262  16.375000      435  27.187500        3421.0  213.812500          24  1.500000          16.0  1.000000              0.602299     1.500000
Aaron Brooks        2005           240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692              0.556845     0.764706
Trent Dilfer        2005           199  18.090909      333  30.272727        2321.0  211.000000          11  1.000000          12.0  1.090909              0.597598     0.916667
Trent Green         2005           317  19.812500      507  31.687500        4014.0  250.875000          17  1.062500          10.0  

Here are the updated dimensions for the table with the new variables added in; of course, nothing changed except the addition of those two variables.

In [23]:
QB_stats.shape

(1456, 12)

Below is a collection of quarterback players that had a total of at least 50 attempts.

In [24]:
QB_stats[QB_stats.attempts['sum'] >= 50].head(50)

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Daunte Culpepper    2005           139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005           302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Jeff Garcia         2005           102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000              0.589595     0.500000
Brett Favre         2005           372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500              0.612850     0.689655
Todd Bouman         2005            68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000              0.557377     0.285714
Brad Johnson        2005           183  16.636364      293  26.636364        1885.0  171.363636          12  1.090909           4.0  0.363636              0.624573     3.000000
Brian Griese        2005           112  18.666667      174  29.000000        1136.0  189.333333           7  1.166667           7.0  1.166667              0.643678     1.000000
Gus Frerotte        2005           257  16.062500      494  30.875000        2996.0  187.250000          18  1.125000          13.0  0.812500              0.520243     1.384615
Kelly Holcomb       2005           155  15.500000      230  23.000000        1509.0  150.900000          10  1.000000           8.0  0.800000              0.673913     1.250000
Mark Brunell        2005           262  16.375000      454  28.375000        3050.0  190.625000          23  1.437500          10.0  0.625000              0.577093     2.300000
Koy Detmer          2005            32   6.400000       56  11.200000         238.0   47.600000           0  0.000000           3.0  0.600000              0.571429     0.000000
Jake Delhomme       2005           262  16.375000      435  27.187500        3421.0  213.812500          24  1.500000          16.0  1.000000              0.602299     1.500000
Aaron Brooks        2005           240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692              0.556845     0.764706
Trent Dilfer        2005           199  18.090909      333  30.272727        2321.0  211.000000          11  1.000000          12.0  1.090909              0.597598     0.916667
Trent Green         2005           317  19.812500      507  31.687500        4014.0  250.875000          17  1.062500          10.0  0.625000              0.625247     1.700000
Drew Bledsoe        2005           300  18.750000      499  31.187500        3639.0  227.437500          23  1.437500          17.0  1.062500              0.601202     1.352941
Matt Hasselbeck     2005           294  18.375000      450  28.125000        3455.0  215.937500          24  1.500000          10.0  0.625000              0.653333     2.400000
Tim Rattay          2005            56  14.000000       97  24.250000         667.0  166.750000           5  1.250000           6.0  1.500000              0.577320     0.833333
Tom Brady           2005           334  20.875000      530  33.125000        4110.0  256.875000          26  1.625000          14.0  0.875000              0.630189     1.857143
Steve McNair        2005           292  20.857143      476  34.000000        3161.0  225.785714          16  1.142857          11.0  

Below are the dimensions of this subsetted data; here, we see that slightly less than two-thirds of all the quarterback players have made a total of at least 50 attempts.

In [25]:
QB_stats[QB_stats.attempts['sum'] >= 50].shape

(966, 12)

Next up is the table sorted by descending completion percentage. Only the first 40 are listed here.

In [26]:
QB_stats_percent_sort=QB_stats.sort_values(by="completion_percentage",ascending=False)
QB_stats_percent_sort[:40]

completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                      
Anthony Wright      2006             3   1.500000        3   1.500000          31.0   15.500000           0  0.000000           0.0  0.000000              1.000000          NaN
Matt Gutierrez      2007             1   0.250000        1   0.250000          15.0    3.750000           0  0.000000           0.0  0.000000              1.000000          NaN
Dennis Dixon        2008             1   1.000000        1   1.000000           3.0    3.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Matt Gutierrez      2009             1   1.000000        1   1.000000           3.0    3.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Brad Smith          2009             1   0.090909        1   0.090909          27.0    2.454545           0  0.000000           0.0  0.000000              1.000000          NaN
Billy Volek         2010             1   0.333333        1   0.333333           8.0    2.666667           0  0.000000           0.0  0.000000              1.000000          NaN
Jordan Palmer       2010             3   3.000000        3   3.000000          18.0   18.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Brian Hoyer         2011             1   0.500000        1   0.500000          22.0   11.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Tyrod Taylor        2011             1   0.500000        1   0.500000          18.0    9.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Derek Anderson      2012             4   2.000000        4   2.000000          58.0   29.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Trent Edwards       2012             2   2.000000        2   2.000000          14.0   14.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Chase Daniel        2012             1   1.000000        1   1.000000          10.0   10.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Colt McCoy          2013             1   0.333333        1   0.333333          13.0    4.333333           0  0.000000           0.0  0.000000              1.000000          NaN
Tarvaris Jackson    2014             1   1.000000        1   1.000000           0.0    0.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Matt Moore          2015             1   1.000000        1   1.000000          14.0   14.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Chase Daniel        2015             2   1.000000        2   1.000000           4.0    2.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Scott Tolzien       2015             1   0.500000        1   0.500000           4.0    2.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Ryan Nassib         2015             5   5.000000        5   5.000000          68.0   68.000000           1  1.000000           0.0  0.000000              1.000000          inf
Chase Daniel        2016             1   1.000000        1   1.000000          16.0   16.000000           0  0.000000           0.0  0.000000              1.000000          NaN
Taylor Heinicke     2017             1   1.000000        1   1.000000          10.0   10.000000           0  0.000000           0.0  

Lastly, we have the table sorted by descending passing touchdown to interception ratio. Again, only the first 40 entries are listed.

In [27]:
QB_stats_ratio_sort=QB_stats.sort_values(by="td_int_ratio",ascending=False)
QB_stats_ratio_sort[:40]

completions            attempts            passing_yards             passing_tds           interceptions      completion_percentage td_int_ratio
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum mean                                   
player_display_name season                                                                                                                                                 
Jeff Blake          2005             8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.0              0.888889          inf
Jay Fiedler         2005             8   4.000000       13   6.500000         107.0   53.500000           1  0.500000           0.0  0.0              0.615385          inf
Quinn Gray          2005             8   8.000000       14  14.000000         100.0  100.000000           2  2.000000           0.0  0.0              0.571429          inf
Chris Weinke        2005             7   2.333333       13   4.333333          64.0   21.333333           1  0.333333           0.0  0.0              0.538462          inf
Matt Schaub         2005            33   4.714286       64   9.142857         495.0   70.714286           4  0.571429           0.0  0.0              0.515625          inf
Charlie Batch       2006            30   4.285714       52   7.428571         477.0   68.142857           5  0.714286           0.0  0.0              0.576923          inf
Vinny Testaverde    2006             2   0.666667        3   1.000000          29.0    9.666667           1  0.333333           0.0  0.0              0.666667          inf
A.J. Feeley         2006            26  13.000000       38  19.000000         342.0  171.000000           3  1.500000           0.0  0.0              0.684211          inf
Todd Collins        2007            67  16.750000      105  26.250000         888.0  222.000000           5  1.250000           0.0  0.0              0.638095          inf
Chris Weinke        2007            13  13.000000       22  22.000000         104.0  104.000000           1  1.000000           0.0  0.0              0.590909          inf
Craig Nall          2007             7   7.000000       15  15.000000          88.0   88.000000           1  1.000000           0.0  0.0              0.466667          inf
Jim Sorgi           2007            18   4.500000       36   9.000000         132.0   33.000000           1  0.250000           0.0  0.0              0.500000          inf
Aaron Rodgers       2007            20  10.000000       28  14.000000         218.0  109.000000           1  0.500000           0.0  0.0              0.714286          inf
Troy Smith          2007            40  10.000000       76  19.000000         452.0  113.000000           2  0.500000           0.0  0.0              0.526316          inf
Quinn Gray          2008             7   7.000000        8   8.000000          76.0   76.000000           1  1.000000           0.0  0.0              0.875000          inf
David Carr          2008             9   3.000000       12   4.000000         115.0   38.333333           2  0.666667           0.0  0.0              0.750000          inf
Byron Leftwich      2008            21   4.200000       36   7.200000         303.0   60.600000           2  0.400000           0.0  0.0              0.583333          inf
Drew Stanton        2008             9   3.000000       17   5.666667         119.0   39.666667           1  0.333333           0.0  0.0              0.529412          inf
Troy Smith          2008             3   0.600000        4   0.800000          82.0   16.400000           1  0.200000           0.0  0.0              0.750000          inf
David Carr          2009            21   3.500000       33   5.500000         225.0   37.500000           1  0.166667           0.0  0.0              0.636364          inf
Mike Vick           2009             6   0.500000       13   1.083333  

Now, we'll repeat this entire process using Spark SQL techniques, including reading in the data. Let's define our Spark session first.

In [28]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('ST 554 Project Two').config('spark.sql.ansi.enabled','false').getOrCreate()

Now, we can read in the data using commands from Spark.

In [29]:
spark2_NFL_data=spark.read.load("weekly_nfl_data.csv",format = "csv",inferSchema = "true",header = "true")

In order to see the first few entries for this data set, the `head()` function will not work here since we're not utilizing `pandas` from this point forward. Although a little messy, the `show()` and `take()` methods accomplish the same goal.

In [30]:
spark2_NFL_data.take(5)

[Row(player_id='00-0000003', player_name=None, player_display_name='Abdul-Karim al-Jabbar', position='RB', position_group='RB', headshot_url=None, recent_team='MIA', season=1999, week=1, season_type='REG', opponent_team='DEN', completions=0, attempts=0, passing_yards=0.0, passing_tds=0, interceptions=0.0, sacks=0.0, sack_yards=0.0, sack_fumbles=0, sack_fumbles_lost=0, passing_air_yards=0.0, passing_yards_after_catch=0.0, passing_first_downs=0.0, passing_epa=None, passing_2pt_conversions=0, pacr=None, dakota=None, carries=16, rushing_yards=60.0, rushing_tds=1, rushing_fumbles=0.0, rushing_fumbles_lost=0.0, rushing_first_downs=4.0, rushing_epa=6.248771, rushing_2pt_conversions=0, receptions=1, targets=1, receiving_yards=7.0, receiving_tds=0, receiving_fumbles=0.0, receiving_fumbles_lost=0.0, receiving_air_yards=0.0, receiving_yards_after_catch=0.0, receiving_first_downs=0.0, receiving_epa=0.29237816, receiving_2pt_conversions=0, racr=0.0, target_share=0.05263158, air_yards_share=None, wo

However, we can still use the `columns` attribute like we did with the `pandas` functionality. The output will look a tad different, though.

In [31]:
spark2_NFL_data.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Now, we'll use SQL to get our collection of quarterback stats that we got earlier. Because of this, different functions were used to get what was desired First, we'll deal with collecting the sums for our variables. By default, only the first 20 entries are shown.

In [32]:
sum_QB_stats2=spark2_NFL_data.select('player_display_name','season','week','completions','attempts','passing_yards','passing_tds','interceptions') \
               .filter((spark2_NFL_data.position == "QB") & (spark2_NFL_data.season > 2004) & (spark2_NFL_data.season_type == "REG")) \
               .groupBy(['player_display_name','season']).sum('completions','attempts','passing_yards','passing_tds','interceptions')
sum_QB_stats2.show()

+-------------------+------+----------------+-------------+------------------+----------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+
|      Jake Delhomme|  2006|             263|          431|            2805.0|              17|              11.0|
|       Jake Plummer|  2005|             277|          456|            3366.0|              18|               7.0|
|        Matt Schaub|  2006|              18|           27|             208.0|               1|               2.0|
|        Vince Young|  2006|             184|          356|            2199.0|              12|              13.0|
|      Kerry Collins|  2007|              50|           82|             531.0|               0|               0.0|
|       Todd Collins|  2005|               0|            0|               0.0|  

Next, we'll tackle the averages here. Second verse, similar to the first, so to speak.

In [33]:
average_QB_stats2=spark2_NFL_data.select('player_display_name','season','week','completions','attempts','passing_yards','passing_tds','interceptions') \
               .filter((spark2_NFL_data.position == "QB") & (spark2_NFL_data.season > 2004) & (spark2_NFL_data.season_type == "REG")) \
               .groupBy(['player_display_name','season']).mean('completions','attempts','passing_yards','passing_tds','interceptions')
average_QB_stats2.show()

+-------------------+------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+------------------+------------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|
|       Jake Plummer|  2005|           17.3125|              28.5|           210.375|             1.125|            0.4375|
|        Matt Schaub|  2006|               3.6|               5.4|              41.6|               0.2|               0.4|
|        Vince Young|  2006|12.266666666666667|23.733333333333334|             146.6|               0.8|0.8666666666666667|
|      Kerry Collins|  2007| 8.333333333333334|13.666666666666666|              88.5|               0.0|               0.0|
|       

We can merge our two tables above by using a `join` method. Here, using `take` appeared to be the less jarring function to use, visually speaking.

In [34]:
QB_stats2=sum_QB_stats2.join(average_QB_stats2,on=['player_display_name','season'])
QB_stats2.take(20)

[Row(player_display_name='Jake Delhomme', season=2006, sum(completions)=263, sum(attempts)=431, sum(passing_yards)=2805.0, sum(passing_tds)=17, sum(interceptions)=11.0, avg(completions)=20.23076923076923, avg(attempts)=33.15384615384615, avg(passing_yards)=215.76923076923077, avg(passing_tds)=1.3076923076923077, avg(interceptions)=0.8461538461538461),
 Row(player_display_name='Jake Plummer', season=2005, sum(completions)=277, sum(attempts)=456, sum(passing_yards)=3366.0, sum(passing_tds)=18, sum(interceptions)=7.0, avg(completions)=17.3125, avg(attempts)=28.5, avg(passing_yards)=210.375, avg(passing_tds)=1.125, avg(interceptions)=0.4375),
 Row(player_display_name='Matt Schaub', season=2006, sum(completions)=18, sum(attempts)=27, sum(passing_yards)=208.0, sum(passing_tds)=1, sum(interceptions)=2.0, avg(completions)=3.6, avg(attempts)=5.4, avg(passing_yards)=41.6, avg(passing_tds)=0.2, avg(interceptions)=0.4),
 Row(player_display_name='Vince Young', season=2006, sum(completions)=184, sum

Next, we define our `completion_percentage` and `td_int_ratio` variables using the `withColumn()` method twice.
Below are the first twenty entries with these new variables added via SQL.

In [35]:
updated_QB_stats2=QB_stats2.withColumn('completion_percentage',QB_stats2['sum(completions)']/QB_stats2['sum(attempts)']) \
         .withColumn('td_int_ratio',QB_stats2['sum(passing_tds)']/QB_stats2['sum(interceptions)'])
updated_QB_stats2.take(20)

[Row(player_display_name='Jake Delhomme', season=2006, sum(completions)=263, sum(attempts)=431, sum(passing_yards)=2805.0, sum(passing_tds)=17, sum(interceptions)=11.0, avg(completions)=20.23076923076923, avg(attempts)=33.15384615384615, avg(passing_yards)=215.76923076923077, avg(passing_tds)=1.3076923076923077, avg(interceptions)=0.8461538461538461, completion_percentage=0.6102088167053364, td_int_ratio=1.5454545454545454),
 Row(player_display_name='Jake Plummer', season=2005, sum(completions)=277, sum(attempts)=456, sum(passing_yards)=3366.0, sum(passing_tds)=18, sum(interceptions)=7.0, avg(completions)=17.3125, avg(attempts)=28.5, avg(passing_yards)=210.375, avg(passing_tds)=1.125, avg(interceptions)=0.4375, completion_percentage=0.6074561403508771, td_int_ratio=2.5714285714285716),
 Row(player_display_name='Matt Schaub', season=2006, sum(completions)=18, sum(attempts)=27, sum(passing_yards)=208.0, sum(passing_tds)=1, sum(interceptions)=2.0, avg(completions)=3.6, avg(attempts)=5.4, 

Again, using SQL, here are the first twenty entries for the quarterback players who got a total of at least 50 attempts.

In [36]:
updated_QB_stats2.filter(updated_QB_stats2['sum(attempts)'] >= 50).take(20)

[Row(player_display_name='Jake Delhomme', season=2006, sum(completions)=263, sum(attempts)=431, sum(passing_yards)=2805.0, sum(passing_tds)=17, sum(interceptions)=11.0, avg(completions)=20.23076923076923, avg(attempts)=33.15384615384615, avg(passing_yards)=215.76923076923077, avg(passing_tds)=1.3076923076923077, avg(interceptions)=0.8461538461538461, completion_percentage=0.6102088167053364, td_int_ratio=1.5454545454545454),
 Row(player_display_name='Jake Plummer', season=2005, sum(completions)=277, sum(attempts)=456, sum(passing_yards)=3366.0, sum(passing_tds)=18, sum(interceptions)=7.0, avg(completions)=17.3125, avg(attempts)=28.5, avg(passing_yards)=210.375, avg(passing_tds)=1.125, avg(interceptions)=0.4375, completion_percentage=0.6074561403508771, td_int_ratio=2.5714285714285716),
 Row(player_display_name='Vince Young', season=2006, sum(completions)=184, sum(attempts)=356, sum(passing_yards)=2199.0, sum(passing_tds)=12, sum(interceptions)=13.0, avg(completions)=12.266666666666667,

Here are the top 40 completion percentages. It appears to be nearly the same as before, but with a different ordering due to the `td-int-ratio` being computed differently when using SQL.

In [37]:
updated_QB_stats2.sort('completion_percentage',ascending=False).take(40)

[Row(player_display_name='Anthony Wright', season=2006, sum(completions)=3, sum(attempts)=3, sum(passing_yards)=31.0, sum(passing_tds)=0, sum(interceptions)=0.0, avg(completions)=1.5, avg(attempts)=1.5, avg(passing_yards)=15.5, avg(passing_tds)=0.0, avg(interceptions)=0.0, completion_percentage=1.0, td_int_ratio=None),
 Row(player_display_name='Matt Gutierrez', season=2007, sum(completions)=1, sum(attempts)=1, sum(passing_yards)=15.0, sum(passing_tds)=0, sum(interceptions)=0.0, avg(completions)=0.25, avg(attempts)=0.25, avg(passing_yards)=3.75, avg(passing_tds)=0.0, avg(interceptions)=0.0, completion_percentage=1.0, td_int_ratio=None),
 Row(player_display_name='Matt Gutierrez', season=2009, sum(completions)=1, sum(attempts)=1, sum(passing_yards)=3.0, sum(passing_tds)=0, sum(interceptions)=0.0, avg(completions)=1.0, avg(attempts)=1.0, avg(passing_yards)=3.0, avg(passing_tds)=0.0, avg(interceptions)=0.0, completion_percentage=1.0, td_int_ratio=None),
 Row(player_display_name='Brad Smith'

Lastly, here are the top 40 touchdown-to-interception ratios. As mentioned before, unlike `pandas`, which treated 0/0 as `NaN` and a positive number divided by zero as `inf`, SQL lumped these two quantity types into one category, simple called `None`.

In [38]:
updated_QB_stats2.sort('td_int_ratio',ascending=False).take(40)

[Row(player_display_name='Tom Brady', season=2016, sum(completions)=291, sum(attempts)=432, sum(passing_yards)=3554.0, sum(passing_tds)=28, sum(interceptions)=2.0, avg(completions)=24.25, avg(attempts)=36.0, avg(passing_yards)=296.1666666666667, avg(passing_tds)=2.3333333333333335, avg(interceptions)=0.16666666666666666, completion_percentage=0.6736111111111112, td_int_ratio=14.0),
 Row(player_display_name='Nick Foles', season=2013, sum(completions)=203, sum(attempts)=317, sum(passing_yards)=2891.0, sum(passing_tds)=27, sum(interceptions)=2.0, avg(completions)=15.615384615384615, avg(attempts)=24.384615384615383, avg(passing_yards)=222.3846153846154, avg(passing_tds)=2.076923076923077, avg(interceptions)=0.15384615384615385, completion_percentage=0.6403785488958991, td_int_ratio=13.5),
 Row(player_display_name='Josh McCown', season=2013, sum(completions)=149, sum(attempts)=224, sum(passing_yards)=1829.0, sum(passing_tds)=13, sum(interceptions)=1.0, avg(completions)=18.625, avg(attempts

I think that does it! Thanks for hainging in there!